<div style="display: flex; justify-content: space-between; align-items: flex-start;">
    <div style="text-align: left;">
        <p style="color:#FFD700; font-size: 15px; font-weight: bold; margin-bottom: 1px; text-align: left;">Published on  April 22, 2026</p>
        <h4 style="color:#4B0082; font-weight: bold; text-align: left; margin-top: 6px;">Author: Jocelyn C. Dumlao</h4>
        <p style="font-size: 17px; line-height: 1.7; color: #333; text-align: center; margin-top: 20px;"></p>
        <a href="https://www.linkedin.com/in/jocelyn-dumlao-168921a8/" target="_blank" style="display: inline-block; background-color: #003f88; color: #fff; text-decoration: none; padding: 5px 10px; border-radius: 10px; margin: 15px;">LinkedIn</a>
        <a href="https://github.com/jcdumlao14" target="_blank" style="display: inline-block; background-color: transparent; color: #059c99; text-decoration: none; padding: 5px 10px; border-radius: 10px; margin: 15px; border: 2px solid #007bff;">GitHub</a>
        <a href="https://www.youtube.com/@CogniCraftedMinds" target="_blank" style="display: inline-block; background-color: #ff0054; color: #fff; text-decoration: none; padding: 5px 10px; border-radius: 10px; margin: 15px;">YouTube</a>
        <a href="https://www.kaggle.com/jocelyndumlao" target="_blank" style="display: inline-block; background-color: #3a86ff; color: #fff; text-decoration: none; padding: 5px 10px; border-radius: 10px; margin: 15px;">Kaggle</a>
    </div>
</div>

# <div style="color:black;display:inline-block;border-radius:4px;background-color:#61d5fa ;font-family:Nexa;overflow:hidden"><p style="padding:8px;color:black;overflow:hidden;font-size:85%;letter-spacing:0.5px;margin:0;border: 6px groove #fbd271;"><b> </b>Import Libraries</p></div>


In [ ]:
import os, json, zipfile, math, copy, sys, time, traceback, random, tempfile, importlib.util
import numpy as np
from pathlib import Path
from itertools import product as iprod

import onnx
import onnx.helper as oh
import onnx.numpy_helper as onh
from onnx import TensorProto


# <div style="color:black;display:inline-block;border-radius:4px;background-color:#61d5fa ;font-family:Nexa;overflow:hidden"><p style="padding:8px;color:black;overflow:hidden;font-size:85%;letter-spacing:0.5px;margin:0;border: 6px groove #fbd271;"><b> </b>Environment Setup</p></div>


In [ ]:
!pip install -q onnxruntime onnx-tool

In [ ]:
# Environment Setup 
try:
    import onnxruntime as ort
except ImportError:
    os.system("pip install onnxruntime -q")
    import onnxruntime as ort

# <div style="color:black;display:inline-block;border-radius:4px;background-color:#61d5fa ;font-family:Nexa;overflow:hidden"><p style="padding:8px;color:black;overflow:hidden;font-size:85%;letter-spacing:0.5px;margin:0;border: 6px groove #fbd271;"><b> </b>Configuration </p></div>

In [ ]:
# Configuration 
TASK_DIR = Path("/kaggle/input/competitions/neurogolf-2026")
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
C, H, W = 10, 30, 30
HW = H * W
SAFE_PAD_IDX = HW - 1

# <div style="color:black;display:inline-block;border-radius:4px;background-color:#61d5fa ;font-family:Nexa;overflow:hidden"><p style="padding:8px;color:black;overflow:hidden;font-size:85%;letter-spacing:0.5px;margin:0;border: 6px groove #fbd271;"><b> </b>Scoring & Cost</p></div>

In [ ]:
# SCORING & COST 
def estimate_model_cost(model):
    """Calculates Params + Bytes + MACs. MACs only counted for Conv/Gemm."""
    try:
        total_params = 0
        total_bytes = 0
        tensors = {}
        for init in model.graph.initializer:
            arr = onh.to_array(init)
            tensors[init.name] = arr
            total_params += arr.size
            total_bytes += arr.nbytes
        for node in model.graph.node:
            if node.op_type == "Constant":
                for attr in node.attribute:
                    if attr.t and (attr.t.raw_data or list(attr.t.float_data)):
                        arr = onh.to_array(attr.t)
                        total_params += arr.size
                        total_bytes += arr.nbytes

        total_macs = 0
        for node in model.graph.node:
            if node.op_type == "Conv":
                if len(node.input) >= 2 and node.input[1] in tensors:
                    w = tensors[node.input[1]]
                    if w.ndim == 4:
                        c_out, c_in, kh, kw = w.shape
                        total_macs += c_out * c_in * kh * kw * H * W
            elif node.op_type == "Gemm":
                if len(node.input) >= 2 and node.input[1] in tensors:
                    w = tensors[node.input[1]]
                    total_macs += w.shape[0] * w.shape[1]
        return int(total_params + total_bytes + total_macs)
    except: return 10**9

def check_model_correct(model, pairs):
    try:
        buf = model.SerializeToString()
        sess = ort.InferenceSession(buf, providers=["CPUExecutionProvider"])
        for p in pairs:
            inp = grid_to_tensor(p["input"])
            out_g = np.array(p["output"])
            oh_, ow_ = out_g.shape
            pred = sess.run(None, {"input": inp})[0]
            if tensor_to_grid(pred, oh_, ow_) != p["output"]: return False
        return True
    except: return False


# <div style="color:black;display:inline-block;border-radius:4px;background-color:#61d5fa ;font-family:Nexa;overflow:hidden"><p style="padding:8px;color:black;overflow:hidden;font-size:85%;letter-spacing:0.5px;margin:0;border: 6px groove #fbd271;"><b> </b>Data Utils</p></div>

In [ ]:
# DATA UTILS
def grid_to_tensor(grid):
    g = np.array(grid, dtype=np.int32)
    h, w = g.shape
    t = np.zeros((1, C, H, W), dtype=np.float32)
    for v in range(C): t[0, v, :h, :w] = (g == v).astype(np.float32)
    return t

def tensor_to_grid(t, out_h, out_w):
    slice_ = t[0, :, :out_h, :out_w]
    return np.where(slice_.max(axis=0) < 0.5, 0, slice_.argmax(axis=0)).tolist()


# <div style="color:black;display:inline-block;border-radius:4px;background-color:#61d5fa ;font-family:Nexa;overflow:hidden"><p style="padding:8px;color:black;overflow:hidden;font-size:85%;letter-spacing:0.5px;margin:0;border: 6px groove #fbd271;"><b> </b>Onnx Builders</p></div>

In [ ]:
# ONNX BUILDERS (Gather & Conv)

def make_gather_model(indices, mask=None):
    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    nodes = [
        oh.make_node("Constant", [], ["sh1"], value=onh.from_array(np.array([1,C,HW],dtype=np.int64))),
        oh.make_node("Reshape", ["input", "sh1"], ["flat"]),
        oh.make_node("Constant", [], ["idx"], value=onh.from_array(indices.astype(np.int32))),
        oh.make_node("Gather", ["flat", "idx"], ["gout"], axis=2),
        oh.make_node("Constant", [], ["sh2"], value=onh.from_array(np.array([1,C,H,W],dtype=np.int64))),
        oh.make_node("Reshape", ["gout", "sh2"], ["output"])
    ]
    return oh.make_model(oh.make_graph(nodes, "gather", [X], [Y]), opset_imports=[oh.make_opsetid("", 17)])

def make_conv_model(w, b=None, ks=3):
    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    nodes = [oh.make_node("Constant", [], ["w"], value=onh.from_array(w.astype(np.float32)))]
    inputs = ["input", "w"]
    if b is not None:
        nodes.append(oh.make_node("Constant", [], ["b"], value=onh.from_array(b.astype(np.float32))))
        inputs.append("b")
    nodes.append(oh.make_node("Conv", inputs, ["output"], kernel_shape=[ks,ks], pads=[ks//2]*4))
    return oh.make_model(oh.make_graph(nodes, "conv", [X], [Y]), opset_imports=[oh.make_opsetid("", 17)])


# <div style="color:black;display:inline-block;border-radius:4px;background-color:#61d5fa ;font-family:Nexa;overflow:hidden"><p style="padding:8px;color:black;overflow:hidden;font-size:85%;letter-spacing:0.5px;margin:0;border: 6px groove #fbd271;"><b> </b>Neural Solver</p></div>

In [ ]:
# NEURAL SOLVER (Adam Optimized)
def try_learned_conv(train_pairs, all_pairs, ks=3, use_bias=True, max_steps=1000):
    try:
        import torch
        import torch.nn as nn
    except: return None
    
    x = torch.stack([torch.from_numpy(grid_to_tensor(p["input"])[0]) for p in train_pairs])
    y = torch.stack([torch.from_numpy(grid_to_tensor(p["output"])[0]) for p in train_pairs])
    
    conv = nn.Conv2d(C, C, ks, padding=ks//2, bias=use_bias)
    opt = torch.optim.Adam(conv.parameters(), lr=0.02)
    
    for _ in range(max_steps):
        opt.zero_grad()
        loss = nn.functional.mse_loss(conv(x), y)
        loss.backward(); opt.step()
        if loss.item() < 1e-5: break
    
    if loss.item() > 0.05: return None
    
    w = conv.weight.detach().numpy()
    b = conv.bias.detach().numpy() if use_bias else None
    m = make_conv_model(w, b, ks)
    return m if check_model_correct(m, all_pairs) else None


# <div style="color:black;display:inline-block;border-radius:4px;background-color:#61d5fa ;font-family:Nexa;overflow:hidden"><p style="padding:8px;color:black;overflow:hidden;font-size:85%;letter-spacing:0.5px;margin:0;border: 6px groove #fbd271;"><b> </b>Spatial Gather Solver</p></div>

In [ ]:
# SPATIAL GATHER SOLVER
def try_spatial_mapping(all_pairs):
    """Finds a per-pixel mapping from input to output."""
    in_t = np.stack([grid_to_tensor(p["input"])[0].reshape(C, -1) for p in all_pairs])
    out_t = np.stack([grid_to_tensor(p["output"])[0].reshape(C, -1) for p in all_pairs])
    
    indices = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    for o_idx in range(HW):
        target = out_t[:, :, o_idx]
        if np.all(target == 0): continue
        
        # Search for a matching input pixel signature
        match = -1
        for i_idx in range(HW):
            if np.array_equal(in_t[:, :, i_idx], target):
                match = i_idx; break
        if match == -1: return None
        indices[o_idx] = match
        
    m = make_gather_model(indices)
    return m if check_model_correct(m, all_pairs) else None


# <div style="color:black;display:inline-block;border-radius:4px;background-color:#61d5fa ;font-family:Nexa;overflow:hidden"><p style="padding:8px;color:black;overflow:hidden;font-size:85%;letter-spacing:0.5px;margin:0;border: 6px groove #fbd271;"><b> </b>Multi Stage Solver</p></div>

In [ ]:
# MULTI-STAGE SOLVER
def solve_task(task_data):
    train = task_data["train"]
    all_v = train + task_data.get("test", []) + task_data.get("arc-gen", [])
    
    # 1. Spatial logic (Rotations, Translations, Flips) - Cost ~36,000
    m = try_spatial_mapping(all_v)
    if m: return m
    
    # 2. 1x1 Conv logic (Color swaps/maps) - Cost ~108,000
    m = try_learned_conv(train, all_v, ks=1, max_steps=1200)
    if m: return m
    
    # 3. 3x3 Conv logic (Object detection/Neighbors) - Cost ~846,000
    m = try_learned_conv(train, all_v, ks=3, max_steps=1500)
    if m: return m

    # 4. Try Constant Output (Fallback)
    if len(set(json.dumps(p["output"]) for p in train)) == 1:
        # Construct logic for constant output if needed
        pass
        
    return None

# <div style="color:black;display:inline-block;border-radius:4px;background-color:#61d5fa ;font-family:Nexa;overflow:hidden"><p style="padding:8px;color:black;overflow:hidden;font-size:85%;letter-spacing:0.5px;margin:0;border: 6px groove #fbd271;"><b> </b>Main Execution</p></div>

In [ ]:
# MAIN EXECUTION
if __name__ == "__main__":
    task_files = sorted(TASK_DIR.glob("task*.json"))
    print(f"Found {len(task_files)} tasks")
    
    solved = []
    total_score = 0.0
    start_time = time.time()

    for tp in task_files:
        tn = tp.stem
        try:
            td = json.load(open(tp))
            t0 = time.time()
            model = solve_task(td)
            dt = time.time() - t0
            
            if model:
                cost = estimate_model_cost(model)
                score = max(1, 25 - math.log(max(1, cost)))
                total_score += score
                solved.append(tn)
                
                out_path = OUTPUT_DIR / f"{tn}.onnx"
                with open(out_path, "wb") as f: f.write(model.SerializeToString())
                
                print(f"  {tn}: cost={cost:>10} score={score:.1f} ({dt:.1f}s)")
        except Exception as e:
            continue

    # Final Zip
    onnx_files = list(OUTPUT_DIR.glob("task*.onnx"))
    with zipfile.ZipFile("submission.zip", "w", zipfile.ZIP_DEFLATED) as zf:
        for f in onnx_files: zf.write(f, f.name)

    print(f"\n{len(solved)}/{len(task_files)} solved, score={total_score:.1f}")